# Homework 2

### Evan Barnett

In [1]:
# %pip install datarec-lib boto3 pandas numpy torch transformers

### Imports and constants

In [4]:
import json
import re
from datetime import datetime, timezone
from io import BytesIO
from pathlib import Path
from tempfile import TemporaryDirectory

import boto3
import numpy as np
import pandas as pd
import torch
from botocore.exceptions import ClientError
from datarec.datasets import Movielens
from transformers import AutoModel, AutoTokenizer

# S3 layout for everything this notebook produces
BUCKET = "barnett-evan-lab3"
DATASET_PREFIX = "datasets/movielens-1m"
EMBEDDING_PREFIX_PRE1980 = "intermediate/movielens-1m-bert-pre1980"
EMBEDDING_PREFIX_FULL = "intermediate/movielens-1m-bert-full"
RECOMMENDATION_PREFIX = "outputs/movielens-1m-recommendations"
PERSONAL_PROFILE_PREFIX = "outputs/movielens-1m-personal-profile"

# Flip to True to re-upload everything (dataset, embeddings, recommendations) on the next run
FORCE_OVERWRITE = True

### S3 helpers

In [5]:
def s3_key_exists(s3, bucket, key):
    # head_object is the cheapest way to check existence without downloading
    try:
        s3.head_object(Bucket=bucket, Key=key)
        return True
    except ClientError as e:
        if e.response.get("Error", {}).get("Code") in {"404", "NoSuchKey", "NotFound"}:
            return False
        raise


def list_s3_keys(s3, bucket, prefix):
    # paginate so we don't silently truncate at 1000 keys
    paginator = s3.get_paginator("list_objects_v2")
    return [
        obj["Key"]
        for page in paginator.paginate(Bucket=bucket, Prefix=prefix.rstrip("/") + "/")
        for obj in page.get("Contents", [])
    ]


def find_s3_key_by_filename(s3, bucket, prefix, filename):
    # MovieLens files can land at different depths under the prefix; pick the shallowest match
    matches = [k for k in list_s3_keys(s3, bucket, prefix) if Path(k).name == filename]
    if not matches:
        raise FileNotFoundError(f"Could not find {filename} under s3://{bucket}/{prefix}/")
    return min(matches, key=len)


def s3_get_bytes(s3, bucket, key):
    # Download an S3 object into an in-memory BytesIO so pandas/numpy can read it directly
    return BytesIO(s3.get_object(Bucket=bucket, Key=key)["Body"].read())

### Load MovieLens 1M from S3

In [6]:
def load_movielens_from_s3(bucket=BUCKET, dataset_prefix=DATASET_PREFIX):
    s3 = boto3.client("s3")

    # MovieLens 1M uses "::" separators 
    def read_dat(filename, columns):
        key = find_s3_key_by_filename(s3, bucket, dataset_prefix, filename)
        return pd.read_csv(
            s3_get_bytes(s3, bucket, key),
            sep="::", engine="python", names=columns, encoding="latin-1",
        )

    movies = read_dat("movies.dat", ["movie_id", "title", "genres"])
    ratings = read_dat("ratings.dat", ["user_id", "movie_id", "rating", "timestamp"])
    users = read_dat("users.dat", ["user_id", "gender", "age", "occupation", "zip"])

    # convert to datetime 
    ratings["interaction_time"] = pd.to_datetime(ratings["timestamp"], unit="s", utc=True)
    return movies, ratings, users

### Download MovieLens 1M and upload a clean copy to S3

In [7]:
def upload_movielens_1m_to_s3(bucket=BUCKET, prefix=DATASET_PREFIX, overwrite=False):
    
    s3 = boto3.client("s3")
    prefix = prefix.rstrip("/")
    required = {"ratings.dat", "movies.dat", "users.dat"}

    # sskip download if S3 already has every file we need
    existing = {Path(k).name for k in list_s3_keys(s3, bucket, prefix)}
    if required.issubset(existing) and not overwrite:
        
        print(f"MovieLens 1M already exists at s3://{bucket}/{prefix}/")
        return

    # Use a temp dir so we don't leave the raw archive on the EC2 instance
    with TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        print("Downloading MovieLens 1M with datarec-lib...")
        Movielens(version="1m", folder=str(tmp)).prepare(only_required=False, use_cache=False)

        # datarec-lib nests the files in a subdirectory, so walk the tree to find them
        files = {p.name: p for p in tmp.rglob("*") if p.is_file() and p.name in required}
        missing = required - set(files)
        if missing:
            raise FileNotFoundError(f"Missing expected MovieLens files: {sorted(missing)}")

        for name, path in files.items():
            key = f"{prefix}/{name}"
            print(f"Uploading {name} -> s3://{bucket}/{key}")
            s3.upload_file(str(path), bucket, key)

    print(f"Done. Uploaded MovieLens 1M to s3://{bucket}/{prefix}/")

### Create BERT embeddings for the candidate movie pool

In [8]:
def create_movie_pool_bert_embeddings(
    bucket=BUCKET,
    dataset_prefix=DATASET_PREFIX,
    output_prefix=EMBEDDING_PREFIX_PRE1980,
    max_year=1980,           # set to None to embed every movie in the dataset
    model_name="bert-base-uncased",
    batch_size=32,
    max_length=128,
    overwrite=False,
):
    s3 = boto3.client("s3")
    output_prefix = output_prefix.rstrip("/")
    embeddings_key = f"{output_prefix}/embeddings.npy"

    # don't recompute by default
    if s3_key_exists(s3, bucket, embeddings_key) and not overwrite:
        print(f"Embeddings already exist at s3://{bucket}/{embeddings_key}")
        return

    movies, _, _ = load_movielens_from_s3(bucket, dataset_prefix)

    # pull the year out of titles like "Toy Story (1995)"
    movies["year"] = movies["title"].str.extract(r"\((\d{4})\)").astype(float)
    if max_year is not None:
        movies = movies[movies["year"] <= max_year].copy()
    movies["year"] = movies["year"].astype(int)

    # build move text : "Title: ... Year: ... Genres: ..."
    movies["clean_title"] = movies["title"].str.replace(r"\s*\(\d{4}\)", "", regex=True)
    movies["genre_text"] = movies["genres"].str.replace("|", ", ", regex=False)
    movies["bert_text"] = (
        "Title: " + movies["clean_title"]
        + ". Year: " + movies["year"].astype(str)
        + ". Genres: " + movies["genre_text"] + "."
    )
    movies = movies.reset_index(drop=True)

    print(f"Creating BERT embeddings for {len(movies)} movies")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device).eval()

    # encode in batches to keep peak memory bounded on a CPU-only instance
    chunks = []
    texts = movies["bert_text"].tolist()
    for start in range(0, len(texts), batch_size):
        inputs = tokenizer(
            texts[start:start + batch_size],
            padding=True, truncation=True, max_length=max_length, return_tensors="pt",
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)

        # Mean-pool over the non-padding tokens (CLS alone undersells short titles/genres)
        mask = inputs["attention_mask"].unsqueeze(-1)
        pooled = (outputs.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1)
        chunks.append(pooled.cpu().numpy())

    embeddings = np.vstack(chunks).astype(np.float32)
    movie_ids = movies["movie_id"].to_numpy(dtype=np.int64)

    # description for embeddings.npy
    config = {
        "dataset": "MovieLens 1M",
        "movie_filter": "all movies" if max_year is None else f"year <= {max_year}",
        "model_name": model_name,
        "pooling": "mean over non-padding tokens",
        "num_movies": int(len(movies)),
        "embedding_dim": int(embeddings.shape[1]),
    }

    # stage everything in a temp dir then upload
    with TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        np.save(tmp / "embeddings.npy", embeddings)
        np.save(tmp / "movie_ids.npy", movie_ids)
        movies.to_csv(tmp / "filtered_movies.csv", index=False)
        (tmp / "embedding_config.json").write_text(json.dumps(config, indent=2))
        for name in ("embeddings.npy", "movie_ids.npy", "filtered_movies.csv", "embedding_config.json"):
            s3.upload_file(str(tmp / name), bucket, f"{output_prefix}/{name}")

    print(f"Uploaded embeddings to s3://{bucket}/{output_prefix}/")

### Run offline preprocessing (pre-1980)

In [9]:
upload_movielens_1m_to_s3(overwrite=FORCE_OVERWRITE)

/opt/conda/lib/python3.11/site-packages/datarec/registry/metrics/movielens_1m.yml
ml-1m.zip: 100% (5917549/5917549 bytes)
ml-1m.zip downloaded from https://files.grouplens.org/datasets/movielens/ml-1m.zip
ml-1m.zip: verifying checksum
Checksum verified.
File decompressed in '/tmp/tmp5iwmbkqi/resource'
Resource ratings ready
Resource movies ready
Resource users ready
Uploading movies.dat -> s3://barnett-evan-lab3/datasets/movielens-1m/movies.dat
Uploading ratings.dat -> s3://barnett-evan-lab3/datasets/movielens-1m/ratings.dat
Uploading users.dat -> s3://barnett-evan-lab3/datasets/movielens-1m/users.dat
Done. Uploaded MovieLens 1M to s3://barnett-evan-lab3/datasets/movielens-1m/


In [10]:
create_movie_pool_bert_embeddings(max_year=1980, overwrite=FORCE_OVERWRITE)

Creating BERT embeddings for 887 movies


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Uploaded embeddings to s3://barnett-evan-lab3/intermediate/movielens-1m-bert-pre1980/


### Recommendation logic for cold and top users

In [11]:
def load_embedding_artifacts(bucket, embedding_prefix):
    # Pull the from S3
    s3 = boto3.client("s3")
    prefix = embedding_prefix.rstrip("/")
    
    embeddings = np.load(s3_get_bytes(s3, bucket, f"{prefix}/embeddings.npy"))
    
    movie_ids = np.load(s3_get_bytes(s3, bucket, f"{prefix}/movie_ids.npy"))
    movies = pd.read_csv(s3_get_bytes(s3, bucket, f"{prefix}/filtered_movies.csv"))
    
    return embeddings, movie_ids, movies


def cosine_similarity_to_profile(embeddings, profile):
    # Standard cosine similarity, vectorized
    return embeddings @ profile / (
        np.linalg.norm(embeddings, axis=1) * np.linalg.norm(profile) + 1e-12
    )


def movies_to_records(df):
    # Make a recommendation slice JSON-friendly
    return [
        {
            "movie_id": int(r["movie_id"]),
            "title": r["title"],
            "year": int(r["year"]),
            "genres": r["genres"],
            "score": float(r["score"]),
        }
        for _, r in df.iterrows()
    ]


def recommend_popular_movies(candidate_movies, ratings, n=5, min_count=50):
    # Cold-user recommednation strategy: top mean rating among movies with at least `min_count` ratings
    candidate_ratings = ratings[ratings["movie_id"].isin(set(candidate_movies["movie_id"]))]
    stats = candidate_ratings.groupby("movie_id")["rating"].agg(
        rating_count="count", avg_rating="mean"
    ).reset_index()
    popular = stats[stats["rating_count"] >= min_count]

    return (
        candidate_movies.merge(
            popular[["movie_id", "avg_rating"]].rename(columns={"avg_rating": "score"}),
            on="movie_id",
        )
        .sort_values("score", ascending=False).head(n)
    )


def recommend_by_profile(candidate_movies, embeddings, movie_ids, liked_movie_ids,
                         exclude_movie_ids, n=5):
    # Profile = simple mean of the embeddings of the liked movies
    id_to_idx = {int(m): i for i, m in enumerate(movie_ids)}
    liked_idx = [id_to_idx[int(m)] for m in liked_movie_ids if int(m) in id_to_idx]
    profile = embeddings[liked_idx].mean(axis=0)

    # Score every candidate, then drop anything the user has already seen
    scored = candidate_movies.assign(score=cosine_similarity_to_profile(embeddings, profile))
    return (
        scored[~scored["movie_id"].isin(set(exclude_movie_ids))]
        .sort_values("score", ascending=False).head(n)
    )


def pick_top_user_with_liked_in_pool(rng, ratings, candidate_movie_ids):
    # Top 5% of users by interaction count
    counts = ratings.groupby("user_id").size()
    top_ids = counts[counts >= counts.quantile(0.95)].index.to_numpy()

    # resample until we land on someone with at least one >=4-star rating in the candidate pool
    pool = set(candidate_movie_ids)
    while True:
        user_id = int(rng.choice(top_ids))
        liked = ratings[
            (ratings["user_id"] == user_id)
            & (ratings["rating"] >= 4)
            & (ratings["movie_id"].isin(pool))
        ]
        if not liked.empty:
            return user_id, liked["movie_id"].tolist()


def summarize_user(user_id, ratings, users, candidate_movie_ids=None):
    # Pull demographic and activity stats for inclusion in the recommendation output
    user_ratings = ratings[ratings["user_id"] == user_id]
    user_row = users[users["user_id"] == user_id].iloc[0].to_dict()
    summary = {
        "user_id": int(user_id),
        "gender": user_row["gender"],
        "age": int(user_row["age"]),
        "occupation": int(user_row["occupation"]),
        "zip": str(user_row["zip"]),
        "total_interactions": int(len(user_ratings)),
        "average_rating": float(user_ratings["rating"].mean()),
    }
    # When a candidate pool is provided, also report how much of the user's history is inside it
    if candidate_movie_ids is not None:
        summary["interactions_in_candidate_pool"] = int(
            user_ratings["movie_id"].isin(set(candidate_movie_ids)).sum()
        )
    return summary, user_ratings["interaction_time"].max().isoformat()


def create_recommendation_outputs(
    bucket=BUCKET,
    dataset_prefix=DATASET_PREFIX,
    pre1980_embedding_prefix=EMBEDDING_PREFIX_PRE1980,
    full_embedding_prefix=EMBEDDING_PREFIX_FULL,
    output_prefix=RECOMMENDATION_PREFIX,
    random_state=42,
    overwrite=False,
):
    s3 = boto3.client("s3")
    output_prefix = output_prefix.rstrip("/")
    output_key = f"{output_prefix}/recommendations.json"
    output_csv_key = f"{output_prefix}/recommendations.csv"

    if s3_key_exists(s3, bucket, output_key) and not overwrite:
        print(f"Recommendations already exist at s3://{bucket}/{output_key}")
        return

    # Make sure both embedding pools exist (each call is a no-op if its embeddings.npy is already up there)
    create_movie_pool_bert_embeddings(bucket, dataset_prefix, pre1980_embedding_prefix, max_year=1980, overwrite=overwrite)
    create_movie_pool_bert_embeddings(bucket, dataset_prefix, full_embedding_prefix, max_year=None, overwrite=overwrite)

    _, ratings, users = load_movielens_from_s3(bucket, dataset_prefix)
    rng = np.random.default_rng(random_state)

    cold_summary = {"description": "No historical ratings or interactions are available for this user."}
    scopes = [
        ("Pre-1980 movies", pre1980_embedding_prefix),
        ("Full MovieLens 1M movies", full_embedding_prefix),
    ]

    # Build the same cold/top-user recommendations against each candidate pool
    results = []
    for scope_name, prefix in scopes:
        embeddings, movie_ids, movies = load_embedding_artifacts(bucket, prefix)

        results.append({
            "Dataset_Scope": scope_name,
            "User_Type": "Cold user",
            "Last_Interaction_Time": None,
            "User_Summary": cold_summary,
            "Recommended_Movies": movies_to_records(
                recommend_popular_movies(movies, ratings, n=5)
            ),
        })

        # Pick a top user per scope so we always land on someone with liked movies in the pool
        top_user_id, liked_ids = pick_top_user_with_liked_in_pool(rng, ratings, movie_ids)
        summary, last_time = summarize_user(top_user_id, ratings, users, movie_ids)
        seen_ids = ratings.loc[ratings["user_id"] == top_user_id, "movie_id"].tolist()

        results.append({
            "Dataset_Scope": scope_name,
            "User_Type": "Top user",
            "Last_Interaction_Time": last_time,
            "User_Summary": summary,
            "Recommended_Movies": movies_to_records(
                recommend_by_profile(movies, embeddings, movie_ids, liked_ids, seen_ids, n=5)
            ),
        })

    # Save both as JSON and CSV
    with TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        (tmp / "recommendations.json").write_text(json.dumps(results, indent=2))
        pd.DataFrame([
            {
                "Dataset_Scope": r["Dataset_Scope"],
                "User_Type": r["User_Type"],
                "Last_Interaction_Time": r["Last_Interaction_Time"],
                "User_Summary": json.dumps(r["User_Summary"]),
                "Recommended_Movies": json.dumps(r["Recommended_Movies"]),
            }
            for r in results
        ]).to_csv(tmp / "recommendations.csv", index=False)
        s3.upload_file(str(tmp / "recommendations.json"), bucket, output_key)
        s3.upload_file(str(tmp / "recommendations.csv"), bucket, output_csv_key)

    print(f"Saved recommendations to s3://{bucket}/{output_key}")
    return results

### Run recommendation generation

In [12]:
recommendations = create_recommendation_outputs(overwrite=FORCE_OVERWRITE)
recommendations

Creating BERT embeddings for 887 movies


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Uploaded embeddings to s3://barnett-evan-lab3/intermediate/movielens-1m-bert-pre1980/
Creating BERT embeddings for 3883 movies


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Uploaded embeddings to s3://barnett-evan-lab3/intermediate/movielens-1m-bert-full/
Saved recommendations to s3://barnett-evan-lab3/outputs/movielens-1m-recommendations/recommendations.json


[{'Dataset_Scope': 'Pre-1980 movies',
  'User_Type': 'Cold user',
  'Last_Interaction_Time': None,
  'User_Summary': {'description': 'No historical ratings or interactions are available for this user.'},
  'Recommended_Movies': [{'movie_id': 2905,
    'title': 'Sanjuro (1962)',
    'year': 1962,
    'genres': 'Action|Adventure',
    'score': 4.608695652173913},
   {'movie_id': 2019,
    'title': 'Seven Samurai (The Magnificent Seven) (Shichinin no samurai) (1954)',
    'year': 1954,
    'genres': 'Action|Drama',
    'score': 4.560509554140127},
   {'movie_id': 858,
    'title': 'Godfather, The (1972)',
    'year': 1972,
    'genres': 'Action|Crime|Drama',
    'score': 4.524966261808367},
   {'movie_id': 922,
    'title': 'Sunset Blvd. (a.k.a. Sunset Boulevard) (1950)',
    'year': 1950,
    'genres': 'Film-Noir',
    'score': 4.491489361702127},
   {'movie_id': 904,
    'title': 'Rear Window (1954)',
    'year': 1954,
    'genres': 'Mystery|Thriller',
    'score': 4.476190476190476}]},

### Personal profile recommendations

Hand-rate 10 movies, save the profile to S3, and recommend 5 movies from the full MovieLens 1M item set.

In [13]:
# My ratings
MY_MOVIE_RATINGS = pd.DataFrame([
    {"title_query": "Star Wars: Episode IV - A New Hope", "year": 1977, "rating": 5},
    {"title_query": "Star Wars: Episode V - The Empire Strikes Back", "year": 1980, "rating": 5},
    {"title_query": "Raiders of the Lost Ark", "year": 1981, "rating": 5},
    {"title_query": "Godfather, The", "year": 1972, "rating": 5},
    {"title_query": "Pulp Fiction", "year": 1994, "rating": 5},
    {"title_query": "Blade Runner", "year": 1982, "rating": 4},
    {"title_query": "Casablanca", "year": 1942, "rating": 4},
    {"title_query": "Toy Story", "year": 1995, "rating": 4},
    {"title_query": "Jurassic Park", "year": 1993, "rating": 3},
    {"title_query": "Batman & Robin", "year": 1997, "rating": 2},
])


def normalize_title_text(text):
    # Lowercase + strip punctuation
    return re.sub(r"[^a-z0-9]+", " ", str(text).lower()).strip()


def match_profile_ratings_to_movielens(profile_ratings, movies):
    movies = movies.copy()
    movies["normalized_title"] = movies["clean_title"].map(normalize_title_text)

    rows = []
    for _, p in profile_ratings.iterrows():
        query = normalize_title_text(p["title_query"])
        #lock matches to the right release year 
        candidates = movies[movies["year"] == int(p["year"])]
        match = candidates[candidates["normalized_title"] == query]
        
        # Loose substring fallback in case my title differs from MovieLens' phrasing
        if match.empty:
            match = candidates[candidates["normalized_title"].str.contains(query, regex=False)]
        
        if match.empty:
            raise ValueError(f"Could not match: {p['title_query']} ({p['year']})")
        row = match.iloc[0].copy()
        row["profile_rating"] = int(p["rating"])
        rows.append(row)

    return pd.DataFrame(rows).reset_index(drop=True)


def create_personal_profile_recommendations(
    bucket=BUCKET,
    dataset_prefix=DATASET_PREFIX,
    embedding_prefix=EMBEDDING_PREFIX_FULL,
    output_prefix=PERSONAL_PROFILE_PREFIX,
    profile_ratings=MY_MOVIE_RATINGS,
    overwrite=False,
):
    
    s3 = boto3.client("s3")
    output_prefix = output_prefix.rstrip("/")
    
    profile_json_key = f"{output_prefix}/user_profile.json"
    profile_csv_key = f"{output_prefix}/user_profile.csv"
    
    rec_json_key = f"{output_prefix}/personal_recommendations.json"
    rec_csv_key = f"{output_prefix}/personal_recommendations.csv"

    if s3_key_exists(s3, bucket, rec_json_key) and not overwrite:
        print(f"Personal recommendations already exist at s3://{bucket}/{rec_json_key}")
        return

    # Reuse the full embedding pool from earlier
    create_movie_pool_bert_embeddings(bucket, dataset_prefix, embedding_prefix, max_year=None, overwrite=overwrite)
    embeddings, movie_ids, movies = load_embedding_artifacts(bucket, embedding_prefix)

    profile_movies = match_profile_ratings_to_movielens(profile_ratings, movies)

    # use the same strategy strategy as the top-user case
    liked_ids = profile_movies.loc[profile_movies["profile_rating"] >= 4, "movie_id"].tolist()
    recs = recommend_by_profile(
        movies, embeddings, movie_ids,
        liked_movie_ids=liked_ids,
        exclude_movie_ids=profile_movies["movie_id"].tolist(),
        n=5,
    )

    # "Last interaction" is just the time we built the profile, since I have no real history
    last_time = datetime.now(timezone.utc).isoformat()
    user_summary = {
        "description": "Hand-created 10-movie profile for offline recommendation testing.",
        "num_profile_ratings": int(len(profile_movies)),
        "average_profile_rating": float(profile_movies["profile_rating"].mean()),
        "liked_movie_count": int((profile_movies["profile_rating"] >= 4).sum()),
    }

    profile_output = {
        "User_Type": "Personal profile",
        "Last_Interaction_Time": last_time,
        "User_Summary": user_summary,
        "Rated_Movies": [
            {
                "movie_id": int(r["movie_id"]),
                "title": r["title"],
                "year": int(r["year"]),
                "genres": r["genres"],
                "rating": int(r["profile_rating"]),
            }
            for _, r in profile_movies.iterrows()
        ],
    }
    rec_output = {
        "Dataset_Scope": "Full MovieLens 1M movies",
        "User_Type": "Personal profile",
        "Last_Interaction_Time": last_time,
        "User_Summary": user_summary,
        "Recommended_Movies": movies_to_records(recs),
    }

    # Save profile + recommendations as both JSON and CSV
    with TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        (tmp / "user_profile.json").write_text(json.dumps(profile_output, indent=2))
        (tmp / "personal_recommendations.json").write_text(json.dumps(rec_output, indent=2))
        profile_movies[["movie_id", "title", "year", "genres", "profile_rating"]].to_csv(
            tmp / "user_profile.csv", index=False
        )
        pd.DataFrame([{
            "Dataset_Scope": rec_output["Dataset_Scope"],
            "User_Type": rec_output["User_Type"],
            "Last_Interaction_Time": rec_output["Last_Interaction_Time"],
            "User_Summary": json.dumps(rec_output["User_Summary"]),
            "Recommended_Movies": json.dumps(rec_output["Recommended_Movies"]),
        }]).to_csv(tmp / "personal_recommendations.csv", index=False)

        s3.upload_file(str(tmp / "user_profile.json"), bucket, profile_json_key)
        s3.upload_file(str(tmp / "user_profile.csv"), bucket, profile_csv_key)
        s3.upload_file(str(tmp / "personal_recommendations.json"), bucket, rec_json_key)
        s3.upload_file(str(tmp / "personal_recommendations.csv"), bucket, rec_csv_key)

    print(f"Saved profile to s3://{bucket}/{profile_json_key}")
    print(f"Saved recommendations to s3://{bucket}/{rec_json_key}")
    return rec_output

### Run personal profile recommendation generation

In [14]:
personal_recommendations = create_personal_profile_recommendations(overwrite=FORCE_OVERWRITE)
personal_recommendations

Creating BERT embeddings for 3883 movies


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Uploaded embeddings to s3://barnett-evan-lab3/intermediate/movielens-1m-bert-full/
Saved profile to s3://barnett-evan-lab3/outputs/movielens-1m-personal-profile/user_profile.json
Saved recommendations to s3://barnett-evan-lab3/outputs/movielens-1m-personal-profile/personal_recommendations.json


{'Dataset_Scope': 'Full MovieLens 1M movies',
 'User_Type': 'Personal profile',
 'Last_Interaction_Time': '2026-05-06T04:49:52.372379+00:00',
 'User_Summary': {'description': 'Hand-created 10-movie profile for offline recommendation testing.',
  'num_profile_ratings': 10,
  'average_profile_rating': 4.2,
  'liked_movie_count': 8},
 'Recommended_Movies': [{'movie_id': 2367,
   'title': 'King Kong (1976)',
   'year': 1976,
   'genres': 'Action|Adventure|Horror',
   'score': 0.9722082018852234},
  {'movie_id': 1692,
   'title': 'Alien Escape (1995)',
   'year': 1995,
   'genres': 'Horror|Sci-Fi',
   'score': 0.9704194068908691},
  {'movie_id': 2851,
   'title': 'Saturn 3 (1979)',
   'year': 1979,
   'genres': 'Adventure|Sci-Fi|Thriller',
   'score': 0.9703996777534485},
  {'movie_id': 3464,
   'title': 'Solar Crisis (1993)',
   'year': 1993,
   'genres': 'Sci-Fi|Thriller',
   'score': 0.9700204730033875},
  {'movie_id': 3701,
   'title': 'Alien Nation (1988)',
   'year': 1988,
   'genres'